# Upgrade-Strategien

Rolling Update ersetzt Pods schrittweise; Blue-Green schaltet zwischen zwei vollständigen Versionen um (siehe Kapitel 03), Canary verteilt einen Teil des Traffics auf die neue Version (Kapitel 06). Die gleichzeitige Nutzung unterschiedlicher Versionen setzt fachliche Kompatibilität voraus. Die historischen NGINX-Tags dienen ausschließlich der Demonstration, nicht dem produktiven Einsatz.

## Voraussetzungen

Python-Kernel, kubectl und K3s-Trainingscluster. Aus diesem Kapitelverzeichnis ausführen. Die verwendeten Demo-Ressourcen müssen frei sein. Aktiven Kontext und Namespace prüfen.

In [ ]:
%%bash
set -eo pipefail
kubectl config current-context
NS=$(kubectl config view --minify -o jsonpath='{..namespace}')
test "${NS:-default}" = default
kubectl get nodes

## 1. Deployment bereitstellen

maxSurge erlaubt einen zusätzlichen Pod, maxUnavailable: 0 verlangt während des Updates die gewünschte Anzahl verfügbarer Pods. Readiness entscheidet über die Aufnahme in Service-Backends.

In [ ]:
%%bash
set -eo pipefail
cat deployment.yml

In [ ]:
%%bash
set -eo pipefail
kubectl apply -f deployment.yml
kubectl rollout status deployment/upgrade-demo --timeout=180s
kubectl get rs,pods -l app=upgrade-demo

## 2. Rolling Update

Das Image im Pod-Template ändern. Ein neues ReplicaSet entsteht. In einem zweiten Terminal kann kubectl get pods -l app=upgrade-demo -w ausgeführt werden.

In [ ]:
%%bash
set -eo pipefail
kubectl set image deployment/upgrade-demo web=nginx:1.28-alpine
kubectl get rs,pods -l app=upgrade-demo
kubectl rollout status deployment/upgrade-demo --timeout=180s
kubectl rollout history deployment/upgrade-demo

## 3. Rollback

Kubernetes führt bei einem fehlgeschlagenen Deployment-Rollout nicht automatisch einen Rollback durch. rollout undo stellt ein früheres Pod-Template wieder her, nicht etwa Datenbankdaten.

In [ ]:
%%bash
set -eo pipefail
kubectl rollout undo deployment/upgrade-demo
kubectl rollout status deployment/upgrade-demo --timeout=180s
kubectl rollout history deployment/upgrade-demo
kubectl get deployment upgrade-demo -o jsonpath='{.spec.template.spec.containers[0].image}'
echo

## 4. Automatische Skalierung

HPA verändert die Replikazahl anhand von Metriken; CPU-Auslastung relativ zu Requests setzt CPU-Requests und eine Metrics API voraus. VPA empfiehlt bzw. verändert Requests (zusätzlicher Controller, nicht standardmäßig installiert). Cluster Autoscaler verändert die Node-Anzahl in unterstützten Umgebungen. Hier nur Überblick.

In [ ]:
%%bash
set -eo pipefail
kubectl get hpa
kubectl get apiservice v1beta1.metrics.k8s.io --ignore-not-found

## 5. Aufräumen

Demo-Ressourcen entfernen.

In [ ]:
%%bash
set -eo pipefail
kubectl delete -f deployment.yml
kubectl get deployment,svc upgrade-demo --ignore-not-found

## Fazit

Rolling Updates und Rollbacks verändern Pod-Templates; sie ersetzen keine fachliche Migrationsstrategie.